In [ ]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer,HashingVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.linear_model import SGDClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier,BaggingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import make_scorer, accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
import numpy as np
from lightgbm import LGBMClassifier,plot_importance,early_stopping,plot_tree
from nltk.tokenize import word_tokenize,sent_tokenize
grid = False

In [ ]:
test_es = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/test_essays.csv')

train1 = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/train_essays.csv')
train1 = train1[['text','generated']]

train2 = pd.read_csv('/kaggle/input/llm-make-7-prompt-train-dataset-v4/train_essays_RDizzl3_seven_v2.csv')
train2.columns = ['text','generated']
train2 = train2[train2.generated==1]

train3 = pd.read_csv('/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv')[['source_text']]
train3['generated']=1
train3.columns = ['text','generated']

train4 = pd.read_csv('/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv')
train4 = train4[train4.RDizzl3_seven == True].reset_index(drop=True)
train4 = train4.rename(columns={'label':'generated'})
train4 = train4[['text','generated']]
train4 = train4[train4.generated==1]

train5 = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text-dataset/Training_Essay_Data.csv')
train5 = train5[train5.generated==1]


train6 = pd.read_csv('/kaggle/input/llm-generated-essays/ai_generated_train_essays.csv')[['text','generated']]

train7 = pd.read_csv('/kaggle/input/llm-generated-essays/ai_generated_train_essays_gpt-4.csv')[['text','generated']]

train8 = pd.read_csv('/kaggle/input/llm-generated-essay-using-palm-from-google-gen-ai/LLM_generated_essay_PaLM.csv')[['text','generated']]

train9 = pd.read_csv('/kaggle/input/daigt-proper-train-dataset/train_drcat_04.csv')
train9 = train9.rename(columns={'label':'generated'})
train9 = train9[['text','generated']]

train10 = pd.read_csv('/kaggle/input/daigt-data-llama-70b-and-falcon180b/llama_70b_v2.csv')
train10 = train10.rename(columns={'generated_text':'text'})
train10 = train10[['text','generated']]
train10 = train10[train10.generated==1]

train = pd.concat([train1,train2,train3,train4,train5,train6,train7,train8,train9,train10])
submisson = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/sample_submission.csv')
train['text'] = train.text.str.replace('\n','')

In [ ]:
def upsample(df:pd.DataFrame, label_col_name:str) -> pd.DataFrame:
    # find the number of observations in the smallest group
    nmin = df[label_col_name].value_counts().max()
    return (df
            # split the dataframe per group
            .groupby(label_col_name)
            # sample nmin observations from each group
            .apply(lambda x: x.sample(nmin,replace=True))
            # recombine the dataframes
            .reset_index(drop=True)
            )

In [ ]:
def compute_features(data):
    sympols = '!@#$%^&*(){}><L:"][~=-_+'
    data['no_of_ptren'] = data.text.apply(lambda x:x.count('(') + x.count(')'))
    data['no_of_ques'] = data.text.apply(lambda x:x.count('?') + x.count('؟'))
    def a(x):
        c=0
        for s in sympols:
            c=c+x.count(s)
        return c
    data['no_of_symps'] = data.text.apply(a)
    data['no_of_words'] = data.text.apply(lambda x : len(word_tokenize(x)))
    data['no_of_parag'] = data.text.apply(lambda x:len(x.split('\n')))
    data['no_of_sen'] = data.text.apply(lambda x:len(sent_tokenize(x)))
    data['len'] = data.text.apply(len)
    data['mean_word_length'] = data.len/data.no_of_words
    data['no_unq_words'] = data.text.apply(lambda x : len(set(word_tokenize(x))))
    data = data[['no_of_ptren','no_of_ques','no_of_symps','no_of_words','no_of_parag','no_of_sen','len','mean_word_length','no_unq_words']].values
    return data

In [ ]:
train.head()

In [ ]:
# train['input_text'] = train['prompt_name']+' - '+train['instructions']+' - '+train['source_text']+' - '+train['text']

In [ ]:
train.generated.value_counts()

In [ ]:
train = train.drop_duplicates()

In [ ]:
train = upsample(train,'generated')

In [ ]:
train.generated.value_counts()

In [ ]:
# X = compute_features(train)
# y = train.generated.values
# X_test = compute_features(test_es)

In [ ]:
# lbgm = LGBMClassifier(n_estimators=10000,max_depth=15,num_leaves=40,learning_rate=0.15,n_jobs=-1,objective='binary')

In [ ]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.02, random_state=42)

In [ ]:
# lbgm.fit(X_train,y_train,eval_set=(X_val,y_val),callbacks=[early_stopping(1000)])

In [ ]:
# preds_test = lbgm.predict_proba(X_test)[:,1]

In [ ]:
# plot_tree(lbgm,figsize=(70,70))

In [ ]:
# plot_importance(lbgm)

In [ ]:
# lbgm.best_score_

In [ ]:
# lbgm.score(X_val,y_val)

In [ ]:
# sgd = SGDClassifier(max_iter=10000, tol=1e-3, loss = "modified_huber")

In [ ]:
vec = TfidfVectorizer(ngram_range=(1,3),sublinear_tf=True)
trans = pd.concat([train[['text']],test_es[['text']]])
y = train.generated.values
X = vec.fit_transform(trans.text.values)

In [ ]:
# sgd.fit(X,y)

In [ ]:
# preds_test2 = sgd.predict_proba(X_test)[:,1]

In [ ]:
# preds_test = (preds_test2*0.95)+(preds_test*0.05)

# MODELS

In [ ]:
lr = LogisticRegression(max_iter=50000,solver="liblinear")
sgd = SGDClassifier(max_iter=50000, tol=1e-3, loss="modified_huber")
voter = VotingClassifier(estimators=[
                                    ('lr', lr), 
                                    ('sgd', sgd), 
                                    #('rf', rf),
                                    #('nb', nb),
                                    #('grad_boost',grad_boost),
                                    #('ada',ada),
                                    #('xg'bst)
                                    #('svm',svm)
                                       ],
                            weights=[0.05,0.95],
                            voting='soft')

In [ ]:
# voter.fit(X[:train.shape[0]],y)
# preds_test = voter.predict_proba(X[train.shape[0]:])[:,1]

In [ ]:
# svm = SVC(probability=True,gamma='auto')
# svm = BaggingClassifier(svm, max_samples=1.0 / 10, n_estimators=10)
# grad_boost = GradientBoostingClassifier()
# lr = LogisticRegression(solver="liblinear")
# #sgd = SGDClassifier(loss = 'log_loss',class_weight = 'balanced')
# #ada = AdaBoostClassifier(estimator = sgd)
# bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')

# ada = AdaBoostClassifier(estimator = sgd)
# rf = RandomForestClassifier()
# nb = MultinomialNB()
# models = [grad_boost,lr,bst,sgd,ada,rf,nb]

In [ ]:
if not grid:
    voter.fit(X[:train.shape[0]],y)
    preds_test = voter.predict_proba(X[train.shape[0]:])[:,1]
else:
    weights = np.linspace(0, 1, 40)
    weight_combinations = [(w,1-w) for w in weights]
    # Define the parameter grid
    param_grid = {'weights': weight_combinations}


    # Define a scorer, for example, accuracy
    scorer = make_scorer(roc_auc_score)

    # Initialize GridSearchCV
    grid_search = GridSearchCV(estimator=voter, 
                       param_grid=param_grid, 
                       scoring=scorer, 
                       cv=5)

    # Fit the grid search to the data
    grid_search.fit(X[:train.shape[0]], y)

    # Find the best parameters
    best_weights = grid_search.best_params_['weights']
    print(f"Best Weights: {best_weights}")
    preds_test = grid_search.predict_proba(X[train.shape[0]:])[:, 1]

In [ ]:
# import gc
# hyp = hyperparameters['svm']
# for class_weight in hyp['class_weight']:
#     for gamma in hyp['gamma']:
#         for kernel in hyp['kernel']:
#             for C in hyp['C']:
#                 svm = SVC(probability=True,class_weight=class_weight,gamma=gamma,kernel=kernel,C=C)
#                 model = svm
#                 model.fit(X_train, y_train)
#                 val_preds = model.predict(X_val)
#                 fpr, tpr, thresholds = metrics.roc_curve(y_val, val_preds)
#                 auc = metrics.auc(fpr, tpr)
#                 print('class_weight = ',class_weight,' - ','gamma= ',gamma,' - ','kernel= ',kernel,' - ','C= ',C,' ==>> AUC = ',auc)

# hyp = hyperparameters['sgd']
# vectorizer1 = TfidfVectorizer(ngram_range=(1,3))
# vectorizer2 = CountVectorizer(ngram_range=(1,3))
# vectorizers = {'tfidf': vectorizer1,'CV':vectorizer2}
# for key,vec in vectorizers.items():
#     X1 = vec.fit_transform(X)
#     X_train, X_val, y_train, y_val = train_test_split(X1, y, test_size=0.1, random_state=42)
#     for loss in hyp['loss']:
#         for penalty in hyp['penalty']:
#             sgd = SGDClassifier(loss = loss,penalty=penalty,class_weight = 'balanced')
#             ada = AdaBoostClassifier(estimator = sgd)
#             model = ada
#             model.fit(X_train, y_train)
#             val_preds = model.predict(X_val)
#             fpr, tpr, thresholds = metrics.roc_curve(y_val, val_preds)
#             auc = metrics.auc(fpr, tpr)
#             print(key,'-',loss,'-',penalty,' ==>> AUC = ',auc)

# hyp = hyperparameters['logistic_regression']
# vectorizer1 = TfidfVectorizer(ngram_range=(1,3))
# vectorizer2 = CountVectorizer(ngram_range=(1,3))
# vectorizers = {'tfidf': vectorizer1,'CV':vectorizer2}
# for key,vec in vectorizers.items():
#     X1 = vec.fit_transform(X)
#     X_train, X_val, y_train, y_val = train_test_split(X1, y, test_size=0.1, random_state=42)
#     for penalty in hyp['penalty']:
#         for solver in hyp['solver']:
#             if solver in ['lbfgs','newton-cg','newton-cholesky','sag'] and penalty not in ['l2',None]:
#                 continue
#             if solver == 'liblinear' and penalty not in ['l2','l1']:
#                 continue
#             lr = LogisticRegression(solver=solver,penalty=penalty,class_weight='balanced',l1_ratio=0.5)
#             model = lr
#             model.fit(X_train, y_train)
#             val_preds = model.predict(X_val)
#             fpr, tpr, thresholds = metrics.roc_curve(y_val, val_preds)
#             auc = metrics.auc(fpr, tpr)
#             f1 = metrics.f1_score(y_val, val_preds)
#             print(key,'-',penalty,'-',solver,' ==>> AUC = ',auc,' ==>> F1 score = ',f1)
#             gc.collect()

In [ ]:
# vectorizer1 = TfidfVectorizer(ngram_range=(1,3),,sublinear_tf=True)
#vectorizer2 = CountVectorizer(ngram_range=(1,3),)
# vec = vectorizer1
# X = vec.fit_transform(train.text.values)
# X_test = vec.transform(test_es.text.values)

In [ ]:
# test_preds = 0
# for model in models:
#     model.fit(X, y)
#     test_preds = test_preds + model.predict_proba(X_test)[:,1]
#     print(model)
# test_preds = test_preds / len(models)

In [ ]:
preds_test

In [ ]:
pd.DataFrame({'id':test_es["id"],'generated':preds_test}).to_csv('submission.csv', index=False)